
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# 프롬프트를 넘어서 – 검색 에이전트와 컨텍스트 엔지니어링

## 서론

생성형 AI 애플리케이션이 프로토타입에서 실제 서비스 출시 단계로 전환되면서, 개발자들은 종종 성능 한계에 직면합니다. 아무리 교묘하게 명령어를 다시 써도, 모델이 학습되지 않은 사실을 강제로 알게 하거나, 중요한 데이터가 없을 때 그것이 환각을 일으키는 것을 막을 수 없습니다. 이는 질문의 표현 방식에 집중하는 **'프롬프트 엔지니어링(Prompt Engineering)'** 에서, 모델에 제공되는 환경 전체를 설계하는 **'컨텍스트 엔지니어링(Context Engineering)'** 으로의 근본적인 패러다임 전환을 의미합니다. 이번 강의에서는 이러한 발전 과정을 살펴보고, 프롬프팅의 한계를 정의하는 한편, 구조적 해결책인 '검색 증강 생성(RAG)'을 소개하겠습니다. 아울러 신뢰성과 비용 효율성을 높이기 위해 컨텍스트 창을 정교하게 설계하는 심화 과정까지 함께 알아보겠습니다.

## 수업 목표

- 범위와 목표에 따라 프롬프트 엔지니어링과 컨텍스트 엔지니어링을 구분
- RAG 아키텍처가 필요한 프롬프트의 세 가지 주요 실패 모드를 식별
- RAG 아키텍처의 과제와 이를 해결하기 위한 전략을 설명
- 컨텍스트 엔지니어링의 기본 원리와 에이전트 시스템에서의 중요성 정의
- 모델 성능 최적화를 위해 컨텍스트 엔지니어링의 주요 원칙 적용
- 에이전트 시스템의 비용, 지연 시간 및 성능 관리를 위한 전략 구현

**코스 범위에 대한 주의 사항:** 본 과목은 아키텍처적 인식을 위한 컨텍스트 엔지니어링(Context Engineering) 개념을 소개합니다. 실습 과제는 기본적인 RAG(검색 증강 생성) 구현에 초점을 맞춰 진행될 예정입니다.

## A. 프롬프트 엔지니어링의 기초

복잡한 AI 아키텍처를 구축하기에 앞서, 우리는 먼저 상호작용의 가장 기본 단위인 '프롬프트'를 마스터해야 합니다. 이번 장에서는 모델의 행동을 유도하는 지시어를 작성하는 기술을 살펴보고, 통계적 예측과 진정한 추론 능력의 차이를 구분해 볼 것입니다. 하지만 프롬프트를 다듬어 나갈수록, 모델이 학습 데이터에만 의존해 알 수 있는 지식에는 명확한 한계가 있다는 점을 결국 마주하게 될 것입니다.

### A1. 프롬프트 엔지니어링의 정의

프롬프트 엔지니어링(Prompt Engineering)은 거대 언어 모델(LLM)이 생성하는 결과물을 최적화하기 위해 입력 텍스트(프롬프트)를 정밀하게 다듬는 작업입니다. 이는 지시문 계층에 초점을 맞춘 **전술적** 영역입니다. 모델의 행동과 출력 형식을 유도하기 위해 **퓨샷 프롬프팅(Few-Shot Prompting, 예시 제공)** 이나 **페르소나 채택(Persona Adoption, 역할 부여)** 과 같은 기법을 사용합니다. 이상적인 프롬프트 엔지니어링은 모델을 하나의 추론 엔진으로 취급하여, 사전 학습된 가중치를 효과적으로 활용해 문제를 해결하도록 인도하는 것입니다.

### A2. 추론 vs. 비추론 모델

효과적인 프롬팅은 기본 모델의 역량을 이해하는 것이 필요합니다. 두 가지 주요 범주를 살펴보겠습니다:

- **비추론 모델(예: Llama 3, GPT-4o):** 이 모델들은 통계적 확률을 기반으로 다음 토큰을 예측합니다. 따라서 복잡한 논리를 풀어나가고 성급하게 잘못된 결론을 내리지 않도록, "단계별로 생각해 봐"와 같은 **생각의 사슬(CoT)** 프롬프팅을 통한 명시적인 안내가 필요합니다.

- **추론 모델(예: OpenAI o1):** 이 모델들은 최종 답변을 내놓기 전에 자체적으로 내부적인 생각의 사슬을 생성하도록 학습되었습니다. 따라서 이러한 모델에 수동으로 CoT 프롬프팅을 적용하는 것은 대개 불필요하거나 오히려 역효과를 낳을 수 있습니다. 추론 모델에 대한 컨텍스트 엔지니어링은 **사고 과정** 보다는 **목표** 와 **제약** 을 정의하는 데 중점을 둡니다.

### A3. 프롬팅의 경계

프롬프트 엔지니어링에는 엄격한 경계가 있습니다. 아무리 명령어를 다듬어도 모델의 학습 데이터에 내재된 다음과 같은 한계를 극복할 수 없습니다:

1. **지식 단절:** 모델은 학습 데이터 종료일 이후에 발생한 사건에 대한 질문에 답할 수 없습니다.
   - *예시 프롬프트:* `2025년 노벨 물리학상 수상자는 누구인가요?`

1. **환각:** 외부 참고 없이 구체적인 사실을 요구할 때, 모델은 종종 진실보다 그럴듯함을 우선시하며 인용이나 데이터 포인트를 조작합니다.
   - *예시 프롬프트:* `아보카도가 혈당 수치를 낮춘다는 것을 증명하는 과학적 참고 문헌을 찾아주세요.`

1. **모호성:** 사적인 맥락이 없으면, 모델은 기본적으로 일반적인 해석으로 나아갑니다.
   - *예시 프롬프트:* `레이크하우스 보안을 위한 방법을 설명해 주세요.` (이 경우 Databricks Data Lakehouse 거버넌스가 아닌 물리적 주택 보안에 대한 조언을 촉발합니다.)

## B. Retrieval Augmented Generation (RAG)

프롬프트 엔지니어링이 모델의 응답 방식을 최적화할 수는 있지만, 모델이 무엇을 알고 있는가라는 근본적인 문제는 해결하지 못합니다. 고정된 학습 데이터와 환각 현상(hallucination)의 한계를 극복하려면, 내부 기억에만 의존하던 기존의 아키텍처 접근 방식을 외부 컨텍스트를 활용하는 방식으로 전환해야 합니다. 이번 섹션에서는 모델의 사전 학습된 지식과 사용자의 자체 데이터 간의 격차를 메워주는 핵심 프레임워크인 **검색 증강 생성(RAG, Retrieval Augmented Generation)** 을 소개합니다.

**RAG vs. 검색 에이전트:** 
**RAG**는 특정 도구나 에이전트 로직과 관계없이, 검색과 생성을 결합하는 아키텍처 패턴을 의미합니다. 반면, **검색 에이전트(Retrieval agents)** 는 실제 시스템 내에서 쿼리 라우팅, 검색 조율, 컨텍스트 결합 등을 처리하며 이러한 패턴을 구체적으로 구현하고 실행하는 주체를 말합니다.

### B1. RAG의 정의

앞서 정의한 지식의 한계를 해결하기 위해, 우리는 기억 기반 방식에서 **검색 증강 생성(RAG)** 으로 알려진 컨텍스트 기반 아키텍처로 전환합니다. RAG는 프롬프트에 자체 보유 데이터나 실시간 데이터를 주입함으로써, 모델이 내부 기억 대신 제공된 사실을 바탕으로 답변할 수 있도록 합니다.

RAG 프로세스는 다음과 같은 세 가지 핵심 단계로 구성됩니다.

- **검색(Retrieval):** 시스템은 관련 데이터 청크를 지식 베이스에서 관련 데이터 청크를 검색합니다.(**Databricks AI Search**를 통해 인덱싱됨)
- **증강(Augmentation):** 시스템은 이 청크들을 컨텍스트 창에 주입합니다
- **생성(Generation):** 모델은 주입된 데이터만을 사용하여 답변을 합성합니다

### B2. 컨텍스트 과제

RAG가 지식의 공백 문제를 해결해 주기는 하지만, **'컨텍스트 부패(Context Rot)'** 라는 새로운 과제를 안겨줍니다. 초기 RAG 방식은 개발자가 단순히 방대한 양의 문서를 가져와 프롬프트에 그대로 붙여넣는 식이었기 때문에 실패하는 경우가 많았습니다. 이러한 방식은 모델에 과부하를 주어 다음과 같은 문제를 일으킵니다.

- **컨텍스트 오염(Context Poisoning):** 모델을 혼란스럽게 만드는 무관하거나 모순되는 정보가 포함되는 현상
- **중간 유실(Lost in the Middle):** 모델이 긴 컨텍스트 창의 중간에 묻힌 정보는 무시하고, 맨 앞이나 맨 뒤에 있는 데이터에만 집중하는 경향

이러한 과제들은 전략적인 컨텍스트 관리가 왜 필요한지 잘 보여주며, 이에 대해서는 다음 섹션에서 자세히 살펴보겠습니다.

## C. 컨텍스트 엔지니어링의 원칙

단순히 데이터를 검색하는 것만으로는 부족합니다. 원시 정보를 프롬프트에 그대로 쏟아붓는 것은 명확성보다는 오히려 혼란을 야기하는 경우가 많습니다. 기본적인 RAG(검색 증강 생성) 시스템에서 상용화 단계의 시스템으로 발전함에 따라, 우리는 컨텍스트 창을 단순히 데이터를 담는 수동적인 컨테이너가 아니라 모델의 동작을 적극적으로 형성하는 엔지니어링된 환경으로 다루어야 합니다. 이 섹션에서는 신뢰성과 정확성을 보장하기 위해 정보를 구조화하고, 필터링하며, 근거를 제시하는 방법에 중점을 두고 컨텍스트 엔지니어링의 원칙을 개괄할 것입니다.

### C1. 컨텍스트 환경 정의

**컨텍스트 엔지니어링**은 전체 입력 창의 전략적 설계입니다. 이는 단일 명령어 작성을 넘어 전체 시스템 상태를 관리하는 단계로 나아갑니다. 우리는 **시스템 명령어**, **대화 기록**, **검색된 데이터**, **사용자 제약 조건** 간의 상호작용을 조율하여 모델이 작업을 효과적으로 수행하는 데 필요한 신호를 정확히 갖추도록 합니다.

### C2. 시스템 프롬프트 설계

컨텍스트 엔지니어링 패러다임에서 시스템 프롬프트는 단순한 요청이 아니라 모델이 어떻게 작동해야 하는지 정의하는 행동 프로그램입니다.

효과적인 시스템 프롬프트의 주요 구성 요소는 다음과 같습니다:

- **역할 정의:** 페르소나를 명확하게 정의합니다. (예: "당신은 Databricks 보안 아키텍트입니다.")
- **부정적 제약 조건:** 모델이 하지 말아야 할 행동을 정의합니다. (예: "경쟁사 제품을 언급하지 마십시오" 또는 "명시적으로 요청하지 않는 한 코드를 제공하지 마십시오.")
- **출력 서식:** 다운스트림 애플리케이션이 응답을 결정론적으로 파싱할 수 있도록 구조화된 출력(예: JSON, YAML 또는 마크다운 표)을 강제합니다.

### C3. 엄격한 그라운딩과 청킹

할루시네이션(환각 현상)을 방지하고 정확성을 확보하기 위해 정보 검색(Retrieval) 과정을 엄격하게 통제해야 합니다.

주요 전략은 다음과 같습니다:

- **그라운딩 지침(Grounding Instructions):** 모델이 검색된 컨텍스트 내에서만 답변하도록 명시적인 지침을 사용합니다. 예를 들어, **제공된 컨텍스트 조각만을 바탕으로 답변하세요. 만약 답변이 포함되어 있지 않다면 "해당 정보를 가지고 있지 않습니다'라고 답변하세요"** 와 같은 방식입니다.
- **메타데이터 필터링:** **Unity Catalog** 메타데이터를 활용해 모델에 도달하기 전에 검색을 필터링합니다. 예를 들어, 사용자가 "2024년 수익"에 대해 묻는다면, 시스템은 모델이 2023년 데이터를 못 보도록 `year=2024`에서 청크를 필터링해야 합니다.

### C4. 다중 턴 상태 관리

에이전트와 같이 긴 대화를 다루는 애플리케이션의 경우, 결국 콘텍스트 창(Context Window)이 가득 차게 됩니다. 따라서 이러한 상태를 관리하기 위해 다음과 같은 전략적인 '콘텍스트 엔지니어링(Context Engineering)' 접근법이 필요합니다.

- **요약(Summarization):** 주기적으로 이전 대화 기록을 압축하여 핵심 결정 사항과 사실 위주로 정리합니다.
- **이동 창(Moving Window):** 가장 오래된 메시지를 삭제하여 새로운 정보를 불러올 토큰 공간을 확보합니다.
- **선택적 유지(Selective Persistence):** 어떤 정보(예: 사용자 이름, 현재 프로젝트 ID)를 콘텍스트에 영구적으로 남겨두고, 어떤 정보를 삭제할지 결정합니다.

이러한 전략을 통해 토큰 제한을 넘지 않으면서도 대화에 필요한 맥락을 효과적으로 유지할 수 있습니다.

## D. 맥락 제약 조건: 토큰 예산과 창 제한

가장 우아하게 설계된 맥락조차도 컴퓨트 리소스와 모델 아키텍처의 엄격한 제약에 따릅니다. 애플리케이션을 확장함에 따라, 포괄적인 맥락에 대한 욕구와 토큰 한도, 지연 시간, 운영 비용 등의 현실 사이에서 균형을 맞춰야 합니다. 이 섹션에서는 컨텍스트 윈도우의 경제학을 살펴보며, 응답 품질을 희생하지 않으면서 토큰 예산을 최적화하는 전략을 제시합니다.

### D1. 컨텍스트 Windows 이해하기

모든 모델에는 **컨텍스트 Windows 제한**(예: 8k, 32k 또는 128k 토큰)이 있습니다. 이는 모델이 사용할 수 있는 작업 기억의 절대적인 한계를 나타냅니다.

컨텍스트 창은 다음과 같습니다:

- **입력 토큰:** 모델에 보내는 텍스트 (명령어 + 검색된 문서 + 이력)
- **출력 토큰:** 모델이 생성하는 텍스트

**트레이드오프:** 더 많은 데이터를 검색한 것으로 창을 채울수록 모델의 추론 능력은 저하되고('Lost in the Middle' 현상), 지연 시간이 크게 증가합니다. 전략적 맥락 관리는 성능 유지에 필수적입니다.

### D2. 토큰 경제학과 최적화

맥락(컨텍스트)은 공짜가 아닙니다. Databricks 파운데이션 모델 API(및 기타 제공업체)는 소비된 입력 및 출력 토큰의 양을 기준으로 비용을 청구합니다.

주요 고려사항:

- **비용 관리:** 모든 쿼리마다 50개의 문서를 가져오는 단순한 방식의 RAG 시스템은 토큰 예산을 빠르게 소진하게 됩니다.

**최적화 전략:**

- **적시 검색(Just-in-Time Retrieval):** 채팅을 시작할 때 매뉴얼 전체를 불러오는 대신, 사용자가 관련 질문을 던졌을 때만 특정 섹션을 검색할 수 있는 도구를 에이전트에게 제공합니다.
- **리랭킹(Reranking):** 리랭커 모델을 사용하여 검색된 상위 50개의 청크(chunk)에 점수를 매긴 후, 가장 관련성이 높은 상위 3~5개만 최종 맥락 창(context window)에 주입합니다.

이러한 전략은 풍부한 맥락을 확보하는 것과 비용 효율성 및 성능 간의 균형을 맞추는 데 도움이 됩니다.

## E. 요약

프롬프트 엔지니어링에서 컨텍스트 엔지니어링으로의 전환은 '미시적 최적화'에서 '거시적 아키텍처'로의 근본적인 패러다임 변화를 의미합니다. 프롬프트가 대화의 어조와 형식을 제어할 수는 있지만, 대형 언어 모델(LLM)이 태생적으로 가진 지식의 공백을 메울 수는 없습니다. 검색 증강 생성(RAG) 아키텍처는 외부 데이터를 주입하여 이 문제를 해결하지만, 동시에 컨텍스트 창(Context Window) 관리라는 복잡한 과제를 안겨줍니다. 컨텍스트 엔지니어링은 입력을 철저하게 구조화하고, 답변의 근거 기준을 엄격히 적용하며, 토큰 예산을 효율적으로 관리함으로써 신뢰할 수 있고 비용 효율적인 AI 시스템을 구축해 이러한 명제들을 해결합니다.

**주요 내용:**

1. **검색 에이전트 아키텍처:** 검색 컴포넌트를 사용해 지식 격차를 메우고, 컨텍스트 엔지니어링을 적용해 검색 에이전트의 성능, 신뢰성, 비용을 최적화해야 합니다.
2. **한정된 컨텍스트 자원:** 컨텍스트 창을 하나의 제한된 예산처럼 관리해야 합니다. 필터링과 재정렬(Reranking)을 활용해 모든 개별 토큰의 가치를 극대화하십시오.
3. **근거 기반 답변의 필수화:** 모델에 *오직* 검색된 데이터만 사용하도록 엄격히 지시하고, **Unity Catalog** 메타데이터를 활용하여 데이터의 관련성과 보안을 보장해야 합니다.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>